# Phase 2 Analysis — Focused Tables for Paper §6 (5 metrics)

Produces tables and statistical tests for the paper's Phase 2 section.

**Includes:**
- 3 Phase 1 baselines: E1 WSFT, E5b Expl Entropy, E7 Dec Only
- 3 Phase 2 family representatives: M1v2 A1 (Linear), best of {WDE/WED/EDW/EWD} (Sequential), M3v2 (Juggler)

**Metrics shown (5 — excludes format_compliance which is saturated/meaningless):**
- decision_accuracy
- safety_score
- clinical_correctness
- completeness
- coherence

**Statistical tests:** Per-method paired t-tests vs. best Phase 1 baseline per metric.

**Outputs:**
- `paper_phase2_table.csv` — full per-method table
- `paper_phase2_compact.csv` — compact paper-ready table (best M2 per size)
- `paper_phase2_significance.csv` — p-values
- LaTeX-ready output printed in Cell 7

In [1]:
# Cell 0: Config + load
import os, json
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
# PROJECT_DIR = os.path.expanduser("~/kd_project")
DATA_DIR = os.path.join(PROJECT_DIR, "data")
N_EVAL = 100

# ── Methods that go in the paper Phase 2 table ──
PHASE1_BASELINES = {
    "E1 WSFT":         "e1_wsft_adapted",
    "E5b Expl Entropy":"e5b_explanation_entropy_sft",
    "E7 Dec Only":     "e7_decision_only_sft",
}

M1_LINEAR = {
    "M1 Linear (A1)":  "m1v2_A1",
}
M2_SEQUENTIAL = {
    "M2 Seq WDE":      "m2v2_WDE",
    "M2 Seq WED":      "m2v2_WED",
    "M2 Seq EDW":      "m2v2_EDW",
    "M2 Seq EWD":      "m2v2_EWD",
}
M3_JUGGLER = {
    "M3 Juggler":      "m3v2_juggler",
}

# ── 5 metrics for the paper (excludes format_compliance) ──
PAPER_METRICS = [
    "decision_accuracy",
    "safety_score",
    "clinical_correctness",
    "completeness",
    "coherence",
]
ALL_METRICS = PAPER_METRICS + ["format_compliance"]  # for loading

# Short labels for compact table headers
METRIC_LABELS = {
    "decision_accuracy":    "Dec.Acc",
    "safety_score":         "Safety",
    "clinical_correctness": "Clin.Corr",
    "completeness":         "Comp",
    "coherence":            "Coher",
    "format_compliance":    "Format",
}

SIZE_MAP = {
    "qwen25_1p5b_base": "1.5B",
    "qwen25_3b_base":   "3B",
    "qwen25_7b_base":   "7B",
}
SIZE_ORDER = ["1.5B", "3B", "7B"]

print(f"Phase 1 baselines: {len(PHASE1_BASELINES)}")
print(f"M1 Linear:         {len(M1_LINEAR)}")
print(f"M2 Sequential:     {len(M2_SEQUENTIAL)}")
print(f"M3 Juggler:        {len(M3_JUGGLER)}")
print(f"Metrics in paper:  {PAPER_METRICS}  ← 5 metrics, no format_compliance")

Phase 1 baselines: 3
M1 Linear:         1
M2 Sequential:     4
M3 Juggler:        1
Metrics in paper:  ['decision_accuracy', 'safety_score', 'clinical_correctness', 'completeness', 'coherence']  ← 5 metrics, no format_compliance


In [2]:
# Cell 1: Load all judge files
def find_judge_file(stub):
    for c in [f"judge__{stub}__{N_EVAL}__g31.jsonl", f"judge__{stub}__{N_EVAL}.jsonl"]:
        path = os.path.join(DATA_DIR, c)
        if os.path.exists(path):
            return path
    return None

def load_judge_jsonl(path):
    out = []
    if not path or not os.path.exists(path):
        return out
    for line in open(path):
        try: obj = json.loads(line)
        except: continue
        if obj.get("status") != "ok": continue
        for mn, sc in obj.get("scores", {}).items():
            if isinstance(sc, dict):
                out.append((obj["id"], mn, sc))
    return out

ALL_METHODS = {**PHASE1_BASELINES, **M1_LINEAR, **M2_SEQUENTIAL, **M3_JUGGLER}

rows = []
for display_name, stub in ALL_METHODS.items():
    path = find_judge_file(stub)
    if not path:
        print(f"  ❌ {display_name}: no judge file")
        continue
    n_loaded = 0
    for sid, mn, sc in load_judge_jsonl(path):
        if mn not in SIZE_MAP: continue
        rec = {"method": display_name, "stub": stub, "model": mn,
               "size": SIZE_MAP[mn], "id": sid}
        for c in ALL_METRICS:
            if c in sc:
                rec[c] = float(sc[c])
        rows.append(rec)
        n_loaded += 1
    print(f"  ✅ {display_name:25s}: {n_loaded} score rows")

df = pd.DataFrame(rows)

# Teacher
teacher_path = find_judge_file("teacher")
teacher_rows = []
if teacher_path:
    for sid, mn, sc in load_judge_jsonl(teacher_path):
        if mn == "teacher":
            rec = {"id": sid}
            for c in ALL_METRICS:
                if c in sc:
                    rec[c] = float(sc[c])
            teacher_rows.append(rec)
teacher_df = pd.DataFrame(teacher_rows)
teacher_means = {c: teacher_df[c].mean() if c in teacher_df else 0 for c in ALL_METRICS}
print(f"\n  Teacher: N={len(teacher_df)}")
for c in PAPER_METRICS:
    print(f"    {c:25s}: {teacher_means[c]:.3f}")

  ✅ E1 WSFT                  : 297 score rows
  ✅ E5b Expl Entropy         : 300 score rows
  ✅ E7 Dec Only              : 300 score rows
  ✅ M1 Linear (A1)           : 294 score rows
  ✅ M2 Seq WDE               : 300 score rows
  ✅ M2 Seq WED               : 300 score rows
  ✅ M2 Seq EDW               : 300 score rows
  ✅ M2 Seq EWD               : 300 score rows
  ✅ M3 Juggler               : 297 score rows

  Teacher: N=100
    decision_accuracy        : 4.350
    safety_score             : 4.240
    clinical_correctness     : 4.220
    completeness             : 3.130
    coherence                : 3.650


In [3]:
# Cell 2: Per-method per-size table — all 5 paper metrics
summary = df.groupby(["method","size"])[PAPER_METRICS].mean().round(3)

method_order = list(PHASE1_BASELINES.keys()) + list(M1_LINEAR.keys()) + \
               list(M2_SEQUENTIAL.keys()) + list(M3_JUGGLER.keys())

wide_rows = []
for m in method_order:
    if m not in summary.index.get_level_values(0): continue
    for size in SIZE_ORDER:
        try:
            row = summary.loc[(m, size)]
            rec = {"method": m, "size": size}
            for c in PAPER_METRICS:
                rec[c] = row[c]
            wide_rows.append(rec)
        except KeyError:
            continue

paper_table = pd.DataFrame(wide_rows)
print("="*120)
print(f"  PHASE 2 — full per-method table (5 metrics: {', '.join(PAPER_METRICS)})")
print("="*120)
display(paper_table)

paper_table.to_csv(os.path.join(DATA_DIR, "paper_phase2_table.csv"), index=False)
print(f"\n✅ Saved → paper_phase2_table.csv")

  PHASE 2 — full per-method table (5 metrics: decision_accuracy, safety_score, clinical_correctness, completeness, coherence)


,method,size,decision_accuracy,safety_score,clinical_correctness,completeness,coherence
0,E1 WSFT,1.5B,4.192,2.869,1.424,2.566,3.545
1,E1 WSFT,3B,4.242,3.131,1.949,2.879,3.929
2,E1 WSFT,7B,4.293,3.727,2.909,3.525,4.374
3,E5b Expl Entropy,1.5B,4.150,2.970,1.480,2.600,3.620
4,E5b Expl Entropy,3B,4.300,3.280,2.070,3.000,3.920
5,E5b Expl Entropy,7B,4.350,3.720,2.750,3.570,4.310
6,E7 Dec Only,1.5B,4.350,2.940,1.680,2.570,3.500
7,E7 Dec Only,3B,4.100,3.300,2.240,3.350,4.000
8,E7 Dec Only,7B,4.150,3.740,2.870,3.730,4.340
9,M1 Linear (A1),1.5B,4.235,2.684,1.316,2.388,3.500



✅ Saved → paper_phase2_table.csv


In [4]:
# Cell 3: Identify best M2 ordering per size (using mean of 5 paper metrics)
print("="*100)
print("  M2 SEQUENTIAL — comparing all 4 orderings to identify best per size")
print(f"  (Best = highest mean of 5 paper metrics, excluding format_compliance)")
print("="*100)

m2_methods = list(M2_SEQUENTIAL.keys())
m2_table = paper_table[paper_table["method"].isin(m2_methods)].copy()
m2_table["mean_5metrics"] = m2_table[PAPER_METRICS].mean(axis=1).round(3)
m2_pivot = m2_table.pivot_table(index="method", columns="size", values="mean_5metrics")
print("\nMean of 5 paper metrics per (M2 ordering × size):")
display(m2_pivot.round(3))

best_m2_per_size = {}
print("\n=== Best M2 ordering per size ===")
for size in SIZE_ORDER:
    if size not in m2_pivot.columns: continue
    best_m = m2_pivot[size].idxmax()
    best_v = m2_pivot[size].max()
    best_m2_per_size[size] = best_m
    print(f"  {size}: {best_m} (mean = {best_v:.3f})")

  M2 SEQUENTIAL — comparing all 4 orderings to identify best per size
  (Best = highest mean of 5 paper metrics, excluding format_compliance)

Mean of 5 paper metrics per (M2 ordering × size):


size,1.5B,3B,7B
method,,,
M2 Seq EDW,2.794,3.108,3.656
M2 Seq EWD,2.904,3.198,3.676
M2 Seq WDE,2.790,3.158,3.536
M2 Seq WED,2.860,3.144,3.716



=== Best M2 ordering per size ===
  1.5B: M2 Seq EWD (mean = 2.904)
  3B: M2 Seq EWD (mean = 3.198)
  7B: M2 Seq WED (mean = 3.716)


In [5]:
# Cell 4: COMPACT paper table — one M2 row per size (the best ordering)
compact_rows = []
for size in SIZE_ORDER:
    # Phase 1 baselines (all 3)
    for m in PHASE1_BASELINES.keys():
        sub = paper_table[(paper_table["method"]==m) & (paper_table["size"]==size)]
        if not sub.empty:
            row = sub.iloc[0]
            rec = {"size": size, "phase": "Phase 1", "method": m}
            for c in PAPER_METRICS:
                rec[c] = row[c]
            compact_rows.append(rec)
    # M1 Linear
    for m in M1_LINEAR.keys():
        sub = paper_table[(paper_table["method"]==m) & (paper_table["size"]==size)]
        if not sub.empty:
            row = sub.iloc[0]
            rec = {"size": size, "phase": "Phase 2", "method": m}
            for c in PAPER_METRICS:
                rec[c] = row[c]
            compact_rows.append(rec)
    # M2 Sequential — only the best ordering
    actual_m = best_m2_per_size.get(size)
    if actual_m:
        sub = paper_table[(paper_table["method"]==actual_m) & (paper_table["size"]==size)]
        if not sub.empty:
            row = sub.iloc[0]
            display_method = f"M2 Seq ({actual_m.replace('M2 Seq ','')})"
            rec = {"size": size, "phase": "Phase 2", "method": display_method}
            for c in PAPER_METRICS:
                rec[c] = row[c]
            compact_rows.append(rec)
    # M3 Juggler
    for m in M3_JUGGLER.keys():
        sub = paper_table[(paper_table["method"]==m) & (paper_table["size"]==size)]
        if not sub.empty:
            row = sub.iloc[0]
            rec = {"size": size, "phase": "Phase 2", "method": m}
            for c in PAPER_METRICS:
                rec[c] = row[c]
            compact_rows.append(rec)

compact_df = pd.DataFrame(compact_rows)
print("="*120)
print("  COMPACT PAPER TABLE — best Phase 1 + best of each Phase 2 family per size")
print("="*120)

for size in SIZE_ORDER:
    print(f"\n--- {size} ---")
    sub = compact_df[compact_df["size"]==size].drop(columns=["size"]).reset_index(drop=True)
    display(sub)

# Add teacher row
teacher_row = pd.DataFrame([{
    "size": "—", "phase": "Teacher", "method": "Teacher (Gemini 2.5)",
    **{c: round(teacher_means[c], 3) for c in PAPER_METRICS}
}])

compact_full = pd.concat([teacher_row, compact_df], ignore_index=True)
compact_full.to_csv(os.path.join(DATA_DIR, "paper_phase2_compact.csv"), index=False)
print(f"\n✅ Saved → paper_phase2_compact.csv  (this is the paper table)")

  COMPACT PAPER TABLE — best Phase 1 + best of each Phase 2 family per size

--- 1.5B ---


,phase,method,decision_accuracy,safety_score,clinical_correctness,completeness,coherence
0,Phase 1,E1 WSFT,4.192,2.869,1.424,2.566,3.545
1,Phase 1,E5b Expl Entropy,4.150,2.970,1.480,2.600,3.620
2,Phase 1,E7 Dec Only,4.350,2.940,1.680,2.570,3.500
3,Phase 2,M1 Linear (A1),4.235,2.684,1.316,2.388,3.500
4,Phase 2,M2 Seq (EWD),4.100,2.780,1.450,2.520,3.670
5,Phase 2,M3 Juggler,3.889,2.455,0.990,2.212,3.242



--- 3B ---


,phase,method,decision_accuracy,safety_score,clinical_correctness,completeness,coherence
0,Phase 1,E1 WSFT,4.242,3.131,1.949,2.879,3.929
1,Phase 1,E5b Expl Entropy,4.300,3.280,2.070,3.000,3.920
2,Phase 1,E7 Dec Only,4.100,3.300,2.240,3.350,4.000
3,Phase 2,M1 Linear (A1),4.337,2.980,1.786,2.765,3.714
4,Phase 2,M2 Seq (EWD),4.200,3.220,1.870,2.760,3.940
5,Phase 2,M3 Juggler,4.040,3.030,1.566,2.576,3.737



--- 7B ---


,phase,method,decision_accuracy,safety_score,clinical_correctness,completeness,coherence
0,Phase 1,E1 WSFT,4.293,3.727,2.909,3.525,4.374
1,Phase 1,E5b Expl Entropy,4.350,3.720,2.750,3.570,4.310
2,Phase 1,E7 Dec Only,4.150,3.740,2.870,3.730,4.340
3,Phase 2,M1 Linear (A1),4.235,3.480,2.520,3.235,4.163
4,Phase 2,M2 Seq (WED),4.300,3.690,2.760,3.470,4.360
5,Phase 2,M3 Juggler,4.040,3.404,2.354,3.121,4.222



✅ Saved → paper_phase2_compact.csv  (this is the paper table)


In [6]:
# Cell 5: Paired t-tests vs the best Phase 1 baseline per (size, metric)
def get_q_scores(method_name, size, metric):
    sub = df[(df["method"]==method_name) & (df["size"]==size)]
    return sub.set_index("id")[metric] if not sub.empty else pd.Series(dtype=float)

phase1_methods = list(PHASE1_BASELINES.keys())
phase2_methods = list(M1_LINEAR.keys()) + list(M2_SEQUENTIAL.keys()) + list(M3_JUGGLER.keys())

sig_rows = []
for size in SIZE_ORDER:
    for metric in PAPER_METRICS:
        p1_means = {m: get_q_scores(m, size, metric).mean() for m in phase1_methods}
        p1_means = {m: v for m, v in p1_means.items() if not pd.isna(v)}
        if not p1_means: continue
        best_p1 = max(p1_means, key=p1_means.get)
        best_p1_val = p1_means[best_p1]
        best_p1_scores = get_q_scores(best_p1, size, metric)

        for p2 in phase2_methods:
            p2_scores = get_q_scores(p2, size, metric)
            if p2_scores.empty: continue
            common = best_p1_scores.index.intersection(p2_scores.index)
            if len(common) < 5: continue
            a = p2_scores.loc[common]
            b = best_p1_scores.loc[common]
            try:
                t, p = stats.ttest_rel(a, b)
            except:
                t, p = float('nan'), float('nan')
            delta = a.mean() - b.mean()
            if pd.isna(p):
                sig = ""
            elif p < 0.001: sig = "***"
            elif p < 0.01: sig = "**"
            elif p < 0.05: sig = "*"
            else: sig = "n.s."
            
            sig_rows.append({
                "size": size,
                "metric": metric,
                "phase2_method": p2,
                "phase2_mean": round(a.mean(), 3),
                "best_phase1": best_p1,
                "phase1_mean": round(best_p1_val, 3),
                "delta": round(delta, 3),
                "p_value": round(p, 4) if not pd.isna(p) else "",
                "sig": sig,
                "n": len(common),
            })

sig_df = pd.DataFrame(sig_rows)
print("="*120)
print("  STATISTICAL SIGNIFICANCE — Phase 2 vs best Phase 1 baseline per (size, metric)")
print("  H0: Phase 2 method = best Phase 1 baseline")
print("  Negative delta = Phase 2 underperforms; positive = Phase 2 improves")
print("="*120)
display(sig_df)
sig_df.to_csv(os.path.join(DATA_DIR, "paper_phase2_significance.csv"), index=False)
print(f"\n✅ Saved → paper_phase2_significance.csv")

  STATISTICAL SIGNIFICANCE — Phase 2 vs best Phase 1 baseline per (size, metric)
  H0: Phase 2 method = best Phase 1 baseline
  Negative delta = Phase 2 underperforms; positive = Phase 2 improves


,size,metric,phase2_method,phase2_mean,best_phase1,phase1_mean,delta,p_value,sig,n
0,1.5B,decision_accuracy,M1 Linear (A1),4.235,E7 Dec Only,4.350,-0.153,0.4082,n.s.,98
1,1.5B,decision_accuracy,M2 Seq WDE,4.050,E7 Dec Only,4.350,-0.300,0.1343,n.s.,100
2,1.5B,decision_accuracy,M2 Seq WED,4.050,E7 Dec Only,4.350,-0.300,0.1343,n.s.,100
3,1.5B,decision_accuracy,M2 Seq EDW,3.950,E7 Dec Only,4.350,-0.400,0.0735,n.s.,100
4,1.5B,decision_accuracy,M2 Seq EWD,4.100,E7 Dec Only,4.350,-0.250,0.2270,n.s.,100
5,1.5B,decision_accuracy,M3 Juggler,3.889,E7 Dec Only,4.350,-0.505,0.0176,*,99
6,1.5B,safety_score,M1 Linear (A1),2.684,E5b Expl Entropy,2.970,-0.265,0.0443,*,98
7,1.5B,safety_score,M2 Seq WDE,2.670,E5b Expl Entropy,2.970,-0.300,0.0169,*,100
8,1.5B,safety_score,M2 Seq WED,2.660,E5b Expl Entropy,2.970,-0.310,0.0219,*,100
9,1.5B,safety_score,M2 Seq EDW,2.670,E5b Expl Entropy,2.970,-0.300,0.0262,*,100



✅ Saved → paper_phase2_significance.csv


In [7]:
# Cell 6: Summary — does any Phase 2 method beat Phase 1 significantly?
print("="*120)
print("  SUMMARY — does any Phase 2 method significantly improve any of the 5 paper metrics?")
print("="*120)

improvements = sig_df[(sig_df["delta"] > 0) & (sig_df["sig"].isin(["*","**","***"]))]
if improvements.empty:
    print("\n  ❌ NO Phase 2 method significantly improves over the best Phase 1 baseline")
    print("     on any of the 5 paper metrics, at any model size.")
    print()
    print("  This is the negative result that motivates §7 (form-over-substance).")
else:
    print(f"\n  Found {len(improvements)} significant improvements:")
    display(improvements)

regressions = sig_df[(sig_df["delta"] < 0) & (sig_df["sig"].isin(["*","**","***"]))]
print(f"\n  Significant regressions (Phase 2 worse than Phase 1): {len(regressions)} of {len(sig_df)}")
if not regressions.empty:
    print("\n  Worst regressions:")
    display(regressions.sort_values("delta").head(10))

# Per-family summary
print("\n=== Per-family summary across all (size, metric) comparisons ===")
for family_name, methods in [("M1 Linear", list(M1_LINEAR.keys())),
                              ("M2 Sequential", list(M2_SEQUENTIAL.keys())),
                              ("M3 Juggler", list(M3_JUGGLER.keys()))]:
    fdf = sig_df[sig_df["phase2_method"].isin(methods)]
    if fdf.empty: continue
    n_improve = len(fdf[(fdf["delta"]>0) & (fdf["sig"].isin(["*","**","***"]))])
    n_regress = len(fdf[(fdf["delta"]<0) & (fdf["sig"].isin(["*","**","***"]))])
    n_total = len(fdf)
    avg_delta = fdf["delta"].mean()
    print(f"  {family_name:15s}: {n_total:3d} comparisons, "
          f"{n_improve:2d} improve, {n_regress:2d} regress, "
          f"avg delta = {avg_delta:+.3f}")

# Per-metric summary
print("\n=== Per-metric summary (averaged across all sizes and Phase 2 methods) ===")
for metric in PAPER_METRICS:
    mdf = sig_df[sig_df["metric"]==metric]
    if mdf.empty: continue
    avg_delta = mdf["delta"].mean()
    n_sig_better = len(mdf[(mdf["delta"]>0) & (mdf["sig"].isin(["*","**","***"]))])
    n_sig_worse = len(mdf[(mdf["delta"]<0) & (mdf["sig"].isin(["*","**","***"]))])
    print(f"  {metric:25s}: avg delta = {avg_delta:+.3f}, "
          f"{n_sig_better} sig improve, {n_sig_worse} sig regress")

  SUMMARY — does any Phase 2 method significantly improve any of the 5 paper metrics?

  ❌ NO Phase 2 method significantly improves over the best Phase 1 baseline
     on any of the 5 paper metrics, at any model size.

  This is the negative result that motivates §7 (form-over-substance).

  Significant regressions (Phase 2 worse than Phase 1): 43 of 90

  Worst regressions:


,size,metric,phase2_method,phase2_mean,best_phase1,phase1_mean,delta,p_value,sig,n
53,3B,completeness,M3 Juggler,2.576,E7 Dec Only,3.350,-0.788,0.0000,***,99
17,1.5B,clinical_correctness,M3 Juggler,0.990,E7 Dec Only,1.680,-0.697,0.0000,***,99
47,3B,clinical_correctness,M3 Juggler,1.566,E7 Dec Only,2.240,-0.687,0.0000,***,99
83,7B,completeness,M3 Juggler,3.121,E7 Dec Only,3.730,-0.626,0.0000,***,99
48,3B,completeness,M1 Linear (A1),2.765,E7 Dec Only,3.350,-0.612,0.0000,***,98
52,3B,completeness,M2 Seq EWD,2.760,E7 Dec Only,3.350,-0.590,0.0000,***,100
77,7B,clinical_correctness,M3 Juggler,2.357,E1 WSFT,2.909,-0.531,0.0018,**,98
11,1.5B,safety_score,M3 Juggler,2.455,E5b Expl Entropy,2.970,-0.515,0.0001,***,99
43,3B,clinical_correctness,M2 Seq WDE,1.730,E7 Dec Only,2.240,-0.510,0.0001,***,100
78,7B,completeness,M1 Linear (A1),3.235,E7 Dec Only,3.730,-0.510,0.0001,***,98



=== Per-family summary across all (size, metric) comparisons ===
  M1 Linear      :  15 comparisons,  0 improve,  9 regress, avg delta = -0.279
  M2 Sequential  :  60 comparisons,  0 improve, 21 regress, avg delta = -0.220
  M3 Juggler     :  15 comparisons,  0 improve, 13 regress, avg delta = -0.448

=== Per-metric summary (averaged across all sizes and Phase 2 methods) ===
  decision_accuracy        : avg delta = -0.187, 0 sig improve, 2 sig regress
  safety_score             : avg delta = -0.248, 0 sig improve, 11 sig regress
  clinical_correctness     : avg delta = -0.414, 0 sig improve, 14 sig regress
  completeness             : avg delta = -0.375, 0 sig improve, 13 sig regress
  coherence                : avg delta = -0.116, 0 sig improve, 3 sig regress


In [8]:
# Cell 7: LaTeX-ready table — paste into paper §6.3
# Note: 5 columns of metrics is a wide table — uses small font + bold for best per (size, metric)

print("="*120)
print("  LATEX-READY TABLE — paste into paper §6.3")
print("="*120)
print()
print(r"\begin{table*}[h]")
print(r"\centering")
print(r"\small")
print(r"\caption{Phase 2 multi-objective methods vs Phase 1 baselines on 5 metrics.")
print(r"None of the three Phase 2 families significantly outperforms the best Phase 1 baseline on")
print(r"safety-critical metrics (safety\_score, clinical\_correctness). Bold = best per (size, metric) within Phase 2.")
print(r"$\dagger$ = significant difference vs best Phase 1 baseline (paired t-test, $p<0.05$).}")
print(r"\begin{tabular}{llccccc}")
print(r"\toprule")
print(r"Size & Method & Dec. Acc. & Safety & Clin. Corr. & Comp. & Coher. \\")
print(r"\midrule")

for size in SIZE_ORDER:
    sub = compact_df[compact_df["size"]==size].copy()
    if sub.empty: continue
    
    # Compute "best per metric" within this size — across both Phase 1 and Phase 2 rows
    bests = {c: sub[c].max() for c in PAPER_METRICS}
    
    first = True
    for i, row in sub.iterrows():
        cells = []
        for c in PAPER_METRICS:
            val = row[c]
            cell = f"\\textbf{{{val:.3f}}}" if val == bests[c] else f"{val:.3f}"
            cells.append(cell)
        size_str = size if first else ""
        first = False
        method_str = row['method'].replace('_', r'\_')
        print(f"{size_str} & {method_str} & {' & '.join(cells)} \\\\")
    print(r"\midrule")

# Teacher row
print(f"--- & Teacher (Gemini 2.5) & "
      f"{teacher_means['decision_accuracy']:.3f} & "
      f"{teacher_means['safety_score']:.3f} & "
      f"{teacher_means['clinical_correctness']:.3f} & "
      f"{teacher_means['completeness']:.3f} & "
      f"{teacher_means['coherence']:.3f} \\\\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\label{tab:phase2}")
print(r"\end{table*}")

  LATEX-READY TABLE — paste into paper §6.3

\begin{table*}[h]
\centering
\small
\caption{Phase 2 multi-objective methods vs Phase 1 baselines on 5 metrics.
None of the three Phase 2 families significantly outperforms the best Phase 1 baseline on
safety-critical metrics (safety\_score, clinical\_correctness). Bold = best per (size, metric) within Phase 2.
$\dagger$ = significant difference vs best Phase 1 baseline (paired t-test, $p<0.05$).}
\begin{tabular}{llccccc}
\toprule
Size & Method & Dec. Acc. & Safety & Clin. Corr. & Comp. & Coher. \\
\midrule
1.5B & E1 WSFT & 4.192 & 2.869 & 1.424 & 2.566 & 3.545 \\
 & E5b Expl Entropy & 4.150 & \textbf{2.970} & 1.480 & \textbf{2.600} & 3.620 \\
 & E7 Dec Only & \textbf{4.350} & 2.940 & \textbf{1.680} & 2.570 & 3.500 \\
 & M1 Linear (A1) & 4.235 & 2.684 & 1.316 & 2.388 & 3.500 \\
 & M2 Seq (EWD) & 4.100 & 2.780 & 1.450 & 2.520 & \textbf{3.670} \\
 & M3 Juggler & 3.889 & 2.455 & 0.990 & 2.212 & 3.242 \\
\midrule
3B & E1 WSFT & 4.242 & 3.131 & 1